In [2]:
%load_ext autoreload
%autoreload 2

In [1]:
import sqlite3
from datetime import datetime
from pathlib import Path
import pandas as pd

import arcgis
from arcgis.gis import GIS
import sys

# 1. Bepaal het absolute pad van de bovenliggende map (de project root)
project_root = str(Path.cwd().parent)

# 2. Voeg de project root toe aan sys.path, zodat Python de 'src' map kan vinden
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.utils as utils
import json


# Create and publish surveys

<div class="alert alert-block alert-info">This notebook uses the ArcGIS API for Python. For more information, see the <a href="https://developers.arcgis.com/python/">ArcGIS API for Python documentation and guides</a>.</div>

 The `create()` method is useful for batch creating blank surveys for which you can reassign ownership to another colleague to complete the design and publish. If you already have an XLSForm, you can use the `create()` and `publish()` methods to automate creating and publishing surveys.

 This sample notebook demonstrates how you can automate creating and publishing surveys using ArcGIS API for Python.

In [3]:
# Get credentials from key vault into environment
# Define your local specific paths 
LOCAL_DB = "G:\Mijn Drive\keepass_db.kdbx"
ENTRY_TITLE = "AGOL"

# Run the function
AGOL_USER, AGOL_PASS = utils.load_keepass_credentials(db_path=LOCAL_DB, entry_name=ENTRY_TITLE)

✅ Success: Credentials for 'AGOL' loaded into environment variables!


In [4]:
agol_url = "https://gisservices.inbo.be/portal"
gis = GIS(agol_url, AGOL_USER, AGOL_PASS)

## Update existing survey

In [ ]:
# Instatiate surveymanager object
survey_manager = arcgis.apps.survey123.SurveyManager(gis)

# Path to XLSForm file
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\Survey123_Volledige_Generatie_20260708.xlsx")

# Fetch old survey ID by name
old_survey_id = utils.get_survey_id_by_name(gis, 'LSVI App Test Auto')
print(f"Bestaande survey ({old_survey_id}) aan het updaten...")
    
# Haal de bestaande survey ops
mijn_survey = survey_manager.get(old_survey_id)

# Overschrijf het formulier met het nieuwe Excel-bestand
mijn_survey.publish(
    xlsform=str(xlsform_path),
    schema_changes=True,  # Sta schemawijzigingen toe
)

print("✅ Survey succesvol overschreven!")

# Fetch object ID of new survey
new_survey_id = utils.get_survey_id_by_name(gis, 'LSVI App Test Auto')
print(new_survey_id)

Bestaande survey (dcf78e12e82949159cebe316e8de7893) aan het updaten...


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\arcgis\apps\survey123\_publish_functions.py:1954: UserWarning: Warning: ['[row : 21] Question has no label: {\'name\': \'rpt_habitat_3110\', \'bind\': {\'relevant\': "selected(string(${habitat_keuze}), \'3110\')"}, \'type\': \'begin repeat\', \'control\': {\'jr:count\': \'1\', \'appearance\': \'minimal\'}}', '[row : 50] Question has no label: {\'name\': \'rpt_habitat_3130_aom\', \'bind\': {\'relevant\': "selected(string(${habitat_keuze}), \'3130_aom\')"}, \'type\': \'begin repeat\', \'control\': {\'jr:count\': \'1\', \'appearance\': \'minimal\'}}', '[row : 96] Question has no label: {\'name\': \'rpt_habitat_3130_na\', \'bind\': {\'relevant\': "selected(string(${habitat_keuze}), \'3130_na\')"}, \'type\': \'begin repeat\', \'control\': {\'jr:count\': \'1\', \'appearance\': \'minimal\'}}', '[row : 136] Question has no label: {\'name\': \'rpt_habitat_3140\', \'bind\': {\'relevant\': "selected(string(${habitat_

✅ Survey succesvol overschreven met de nieuwe vragen!


## Create a survey

You can create surveys using ArcGIS API for Python with the `create()` method. The `create()` method creates an empty form item and hosted feature service in the folder supplied to the method or a new folder created with the survey. The output of the `create()` method is a single <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.Survey">`Survey`</a> object. 

Surveys created by the `create()` method create the equivalent of a new unpublished web designer survey. After creating a survey with ArcGIS API for Python, if you don't have an XLSForm or existing survey design, you can use the <a href="https://doc.arcgis.com/en/survey123/browser/create-surveys/publishsurvey.htm">Survey123 web designer</a> to design and publish your survey. 

The first step is to connect to your GIS.

Next, a `SurveyManager` is defined, and a new survey is created. A survey in the Survey Manager is a single instance of a survey project that contains the item information and properties and provides access to the underlying survey dataset. For more information on Survey Manager, see the <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.SurveyManager" target="_blank">API Reference for the ArcGIS API for Python</a>.

To create a survey, use the `create()` method on a `SurveyManager` instance. 

The `create()` method only requires the `title` parameter. By default, the `create()` method will automatically create a folder using the `Survey-{title}` naming convention consistent with other Survey123 publishing apps. Alternatively, if you want to store your survey in a folder that already exists in your content, you can use the `folder` parameter and specify the name or folder ID.

The result of the `create()` method is a <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#survey" target="_blank">Survey</a> object. At this stage, the survey is in a draft state and has no schema or questions. You can use the Survey123 web designer to design your survey and publish from there, use the `publish()` method with an existing XLSForm design, or use a third-party Python package to design your survey directly in Python and call the `publish()` method. 

## Publish a survey

To publish a survey using the `publish()` method, an existing `Survey` object is required. The `Survey` can be an unpublished survey created by the `create()` method or an existing published survey in your content. It can also be an unpublished blank survey created with the Survey123 web designer (any designs will be overwritten by the `publish()` method's required XLSForm.). 

When publishing surveys with the `publish()` method, important considerations are as follows:

- Any survey is treated as a survey published with Survey123 Connect, which means you can no longer use the web designer to edit the portions of the survey that are managed by the XLSForm. For example, the survey title and survey questions cannot be edited. Themes, webhooks, and sharing options can still be edited in the web designer. 
- New surveys will create and use a `_form` view as the submission endpoint, which means you can no longer update the feature service schema in Survey123 Connect. 
- Existing surveys will maintain their submission endpoint.

A `schema_changes` parameter is available to modify the schema of your submission endpoint. For more information on the `schema_changes` parameter please see the <a href="https://developers.arcgis.com/python/api-reference/arcgis.apps.survey123.html#arcgis.apps.survey123.Survey.publish" target="_blank">ArcGIS API for Python</a> documentation.

In this example, the required `xlsform` parameter is set to the path where the XLSForm is located and optional parameters are set: 

- `info` is set to enable the Inbox.
- `enable_delete_protection` is enabled for the form item and its related content. 

In [51]:
# Instatiate surveymanager object
survey_manager = arcgis.apps.survey123.SurveyManager(gis)

# Path to XLSForm file
xlsform_path = Path(r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\Survey123_Volledige_Generatie_20260708.xlsx")

In [55]:
for item in remaining_items:
    print(item)

In [58]:
# We first have to wipe the existing folder and its contents (feature layers, forms, etc.)

# Fetch old survey ID by name before deletion
old_survey_id = utils.get_survey_id_by_name(gis, 'LSVI App Test Auto')
print(f"Bestaande survey ({old_survey_id}) aan het updaten...")

form_item = gis.content.get(old_survey_id)
user = gis.users.get(form_item.owner)
item_title = form_item.title

folder_name, folder_id = utils.browse_folders_for_survey(gis, old_survey_id)
print(folder_name, folder_id)

# Fetch all items residing inside this specific folder
folder_items = user.items(folder=folder_name)

# First remove dependent items: forms, views, maps
for item in folder_items:
    # Check if the item is the absolute root/source Feature Service
    is_source_layer = (item.type == "Feature Service" and "ViewService" not in item.typeKeywords)
    
    if not is_source_layer:
        try:
            print(f"Disabling protection and deleting: {item.title} ({item.type})")
            item.protect(enable=False)  # Lift the protection lock
            item.delete()               # Delete the item
        except Exception as e:
            print(f" [!] Failed to delete dependent item '{item.title}': {e}")

# We query the folder again to grab the leftover source layer(s)
remaining_items = user.items(folder=folder_name)
for item in remaining_items:
    try:
        print(f"Disabling protection and deleting source layer: {item.title} ({item.type})")
        item.protect(enable=False)
        item.delete()
    except Exception as e:
        print(f" [!] Failed to delete source layer '{item.title}': {e}")

# Only delete if it's an actual subfolder and not your root directory
if folder_name and folder_name.lower() != "root":
    try:
        print(f"Removing empty folder structure: '{folder_name}'...")
        gis.content.delete_folder(folder=folder_name, owner=form_item.owner)
        print("Successfully wiped folder and all its contents.")
    except Exception as e:
        print(f" [!] Failed to delete the folder itself: {e}")
else:
    print("Items deleted from root. Skipping folder deletion.")

c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory 'C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmpxx3to48h'>
  _warnings.warn(warn_message, ResourceWarning)
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory 'C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmp9u2xneq1'>
  _warnings.warn(warn_message, ResourceWarning)


Bestaande survey (9cab29dfc32742a7ab3e804493956c14) aan het updaten...
Survey-LSVI App Test Auto d30a938136714f5e8a20d8234ed6b3c4
Disabling protection and deleting: LSVI App Test Auto (Form)
Disabling protection and deleting: LSVI App Test Auto (Web Map)
Disabling protection and deleting source layer: LSVI App Test Auto (Feature Service)
 [!] Failed to delete source layer 'LSVI App Test Auto': Unable to delete item. This service item has a related Service item
(Error Code: 400)
Disabling protection and deleting source layer: LSVI App Test Auto_form (Feature Service)
Removing empty folder structure: 'Survey-LSVI App Test Auto'...


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: delete_folder is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `Folder.delete()` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Successfully wiped folder and all its contents.


In [59]:
# Create new folder for survey
folder_name = "Survey-LSVI App Test Auto"
map_object = gis.content.create_folder(folder=folder_name)
print(f"Succes! De map '{folder_name}' is aangemaakt.")
print(map_object)

# Create new survey in dedicated folder
survey_manager = arcgis.apps.survey123.SurveyManager(gis)
new_survey = survey_manager.create(
    title = "LSVI App Test Auto",
    folder = "Survey-LSVI App Test Auto",
    tags = "LSVI, Survey123, Python",
    summary = "Test Survey123 for LSVI field work.",
    description = "This survey is created for testing the LSVI field work app. We tried to publish this survey automatically from an XLSX form.", 
    thumbnail = r"../inbo_logo.jpg"
)

published_survey = new_survey.publish(
    xlsform=str(xlsform_path), 
    # info=
    #     {"queryInfo": {
    #         "mode": "manual",
    #         "editEnabled": True,
    #         "copyEnabled": False
    #         }
    #     }, 
    enable_delete_protection=False, # We want to be able to delete and overwrite the survey
    enable_sync=True, # Enable sync for offline use,
    thumbnail = r"../inbo_logo.jpg",
    schema_changes=True
    )

# Fetch object ID of new survey
new_survey_id = utils.get_survey_id_by_name(gis, 'LSVI App Test Auto')
print(new_survey_id)

c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\IPython\core\interactiveshell.py:3748: DeprecatedWarning: create_folder is deprecated as of 2.3.0 and has been removed in 3.0.0. Use `gis.content.folders.create` instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Succes! De map 'Survey-LSVI App Test Auto' is aangemaakt.
{'username': 'joost.neujens@inbo.be', 'id': 'c1dca8784eb644a7954f90cf9622612c', 'title': 'Survey-LSVI App Test Auto'}


c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\arcgis\apps\survey123\_survey.py:214: ResourceWarning: unclosed file <_io.BufferedReader name='../inbo_logo.jpg'>
  form_item = folder_obj.add(item_properties=form_properties, text="{}").result()
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\tempfile.py:934: ResourceWarning: Implicitly cleaning up <TemporaryDirectory 'C:\\Users\\JOOST_~1\\AppData\\Local\\Temp\\tmpmnl5b02d'>
  _warnings.warn(warn_message, ResourceWarning)
c:\Users\joost_neujens\AppData\Local\anaconda3\envs\python-gis\Lib\site-packages\arcgis\apps\survey123\_publish_functions.py:1954: UserWarning: Warning: ['[row : 21] Question has no label: {\'bind\': {\'relevant\': "selected(string(${habitat_keuze}), \'3110\')"}, \'type\': \'begin repeat\', \'name\': \'rpt_habitat_3110\', \'control\': {\'appearance\': \'minimal\', \'jr:count\': \'1\'}}', '[row : 50] Question has no label: {\'bind\': {\'relevant\': "selected(string(${ha

93a1bab3786f47db903d5de2b84c098d


### Sync BWK Field Map Web Map With Latest Form ID

In [60]:
# PoC --> zit arcade expressie URL ook in map_item?
# En zou dit aangepast kunnen worden en geupdate kunnen worden? 
import re


FIELD_MAP_ID = '64c1f0bd02344d5ebf41c3dd320615bc'
gis = GIS(agol_url, AGOL_USER, AGOL_PASS)
map_item = gis.content.get(FIELD_MAP_ID)
    
# Get the web map's internal data configuration
map_data = map_item.get_data()

# Convert configuration to a string to easily replace the old ID globally
map_data_str = json.dumps(map_data)

# old_survey_id = "cd5b245d1a154c25b25982d6efa5dfcf"
survey_id_pattern = r"arcgis-survey123://\?itemID=([a-fA-F0-9]{32})"
match = re.search(survey_id_pattern, map_data_str)

if match:
    # Extract the matched group (the old ID)
    old_survey_id = match.group(1)
    print(f"Found existing survey ID in Arcade expression: {old_survey_id}")
    
    if old_survey_id != new_survey_id:
        # Swap the old ID out for the new ID globally across the entire string
        updated_map_data_str = map_data_str.replace(old_survey_id, new_survey_id)
        updated_map_data = json.loads(updated_map_data_str)
        
        # Update the Web Map item with the corrected configuration
        map_item.update(data=updated_map_data)
        print(f"Field Map pop-up links successfully updated to new ID: {new_survey_id}")
    else:
        print("The Web Map is already using the current new_survey_id. No update needed.")
else:
    print("Warning: Could not find an 'arcgis-survey123://?itemID=' link anywhere inside the Web Map configuration.")


Found existing survey ID in Arcade expression: dcf78e12e82949159cebe316e8de7893
Field Map pop-up links successfully updated to new ID: 93a1bab3786f47db903d5de2b84c098d


### Attempt to sync our surveys 

In [ ]:
# # Initialiseer de Survey123 Manager
# survey_manager = survey123.SurveyManager(gis)

# # Make nieuwe Web Map, Feature Layer en Form Item aan.
# def publiceer_nieuwe_survey(xlsform_path):
#     print("Nieuwe survey aanmaken...")
#     nieuwe_survey = survey_manager.create(
#         title="LSVI Survey - Automatisch",
#         folder="LSVI App Test",  # Hier specificeer je jouw AGOL map!
#         tags = "Water surveys, Survey123, Python",
#         summary = "lsvi",
#         description = "Survey automatisch gegenereerd via Python script op basis van vereisten en soortenlijsten in LSVI package. Testen van dynamische vraaggeneratie en koppeling met AGOL.", 
#     )
#     print(f"✅ Succes! Nieuwe survey aangemaakt met ID: {nieuwe_survey}")

#     # En publiceren
#     nieuwe_survey.publish(form=xlsform_path,
#                           info=
#                             {"queryInfo": {
#                                 "mode": "manual",
#                                 "editEnabled": True,
#                                 "copyEnabled": False
#                                 }
#                             }, 
#                         enable_delete_protection=False, # We want to be able to delete and overwrite the survey
#                         )
#     print("✅ Survey succesvol gepubliceerd!")
#     return nieuwe_survey

# # Update bestaande survey
# def update_bestaande_survey(form_item_id, xlsform_path):
#     print(f"Bestaande survey ({form_item_id}) aan het updaten...")
    
#     # Haal de bestaande survey op
#     mijn_survey = survey_manager.get(form_item_id)
    
#     # Overschrijf het formulier met het nieuwe Excel-bestand
#     mijn_survey.update(form=xlsform_path)
#     print("✅ Survey succesvol overschreven met de nieuwe vragen!")

In [ ]:
# form_item_id = "9d86f50905c143c28fc4f419ef575a49"  # Vervang dit door het ID van jouw bestaande survey
# xlsform_path = r"Q:\Projects\PRJ_GIS\lsvi-app-testing\output\Survey123_Volledige_Generatie_20260601.xlsx"
# print(f"Bestaande survey ({form_item_id}) aan het updaten...")

# # Haal de bestaande survey op
# mijn_survey = survey_manager.get(form_item_id)

# # Overschrijf het formulier met het nieuwe Excel-bestand
# mijn_survey.update(form=xlsform_path)
# print("✅ Survey succesvol overschreven met de nieuwe vragen!")
